# SCARS RANDOM Code Truth Swarm Notebook

Build a deterministic actor workflow that uses your local RAG/MCP idea, verifies code paths, protects schemas with Pydantic, runs tests, and writes proof before trusting Gemini/Gemma output.

This is not the job finder itself yet. This is the higher-level system that makes codegen safe.

## Mental model

Do not serve the whole pipeline first.

First build the truth pipeline:

```text
RAG says code exists
↓
Path actor confirms file exists
↓
Schema actor validates shape
↓
Usage actor confirms imports/symbols/paths
↓
Source pack actor writes proof
↓
Gemini/Gemma receives verified pack only
↓
Test actor runs generated code
↓
Proof reporter exports CSV/MD/JSON
```

The forest is the validated knowledge layer: Pydantic schemas + DuckDB/LanceDB/Ray actors + proof exports.

In [1]:
# Cell 1 — Imports and fixed project paths

import os
import sys
import re
import ast
import json
import time
import uuid
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Optional, Literal

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

PROJECT_ROOT = Path(r"C:\WEB CASE STUDY")
PYTHON_EXE = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"
RAG_QUERY_SCRIPT = PROJECT_ROOT / "ADAMSCARMCCOY_QUERY_RAG.PY"

CORRECT_LANCEDB_PATH = Path(
    r"C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\ableton-session-intelligence\lancedb_web_intel_rag"
)

WRONG_EMPTY_E_PATH = Path(
    r"E:\WEB CASE STUDY\Legion-Jacked-Pipeline\ableton-session-intelligence\lancedb_web_intel_rag"
)

TRUTH_OUT = PROJECT_ROOT / "code_truth_exports"
TRUTH_OUT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PYTHON_EXE exists:", PYTHON_EXE.exists(), PYTHON_EXE)
print("RAG_QUERY_SCRIPT exists:", RAG_QUERY_SCRIPT.exists(), RAG_QUERY_SCRIPT)
print("CORRECT_LANCEDB_PATH exists:", CORRECT_LANCEDB_PATH.exists(), CORRECT_LANCEDB_PATH)
print("TRUTH_OUT:", TRUTH_OUT)

PROJECT_ROOT: C:\WEB CASE STUDY
PYTHON_EXE exists: True C:\WEB CASE STUDY\.venv\Scripts\python.exe
RAG_QUERY_SCRIPT exists: True C:\WEB CASE STUDY\ADAMSCARMCCOY_QUERY_RAG.PY
CORRECT_LANCEDB_PATH exists: True C:\STUDIES_BACKUP\Legion-Jacked-Pipeline\ableton-session-intelligence\lancedb_web_intel_rag
TRUTH_OUT: C:\WEB CASE STUDY\code_truth_exports


In [2]:
# Cell 2 — Install/import dependency check, no magic installs unless you uncomment

required = ["ray", "pydantic", "pandas", "pyarrow", "lancedb"]
missing = []

for name in required:
    try:
        __import__(name)
        print(f"ok: {name}")
    except Exception as e:
        print(f"missing/broken: {name} -> {e}")
        missing.append(name)

if missing:
    print("\nMissing packages. Install manually if needed:")
    print(f'"{PYTHON_EXE}" -m pip install ' + " ".join(missing))

ok: ray
ok: pydantic
ok: pandas
ok: pyarrow
ok: lancedb


In [3]:
# Cell 3 — Pydantic schema protection for the Code Truth Swarm

from pydantic import BaseModel, Field, ConfigDict

class RagHit(BaseModel):
    model_config = ConfigDict(extra="allow")
    query: str
    source_file: str
    symbol_name: Optional[str] = None
    distance: Optional[float] = None
    content: Optional[str] = None
    raw_text: Optional[str] = None

class PathProof(BaseModel):
    source_file: str
    normalized_path: str
    exists: bool
    readable: bool
    size_bytes: Optional[int] = None
    modified_at: Optional[str] = None
    contains_symbol: Optional[bool] = None
    symbols_found: list[str] = Field(default_factory=list)
    error: Optional[str] = None

class UsageProof(BaseModel):
    source_file: str
    imports: list[str] = Field(default_factory=list)
    functions: list[str] = Field(default_factory=list)
    classes: list[str] = Field(default_factory=list)
    constants: dict[str, str] = Field(default_factory=dict)
    paths_found: list[str] = Field(default_factory=list)
    lancedb_tables_found: list[str] = Field(default_factory=list)
    risk_flags: list[str] = Field(default_factory=list)
    error: Optional[str] = None

class TruthPack(BaseModel):
    pack_id: str = Field(default_factory=lambda: f"truthpack_{uuid.uuid4().hex[:8]}")
    created_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    queries: list[str]
    rag_hits: list[RagHit]
    path_proofs: list[PathProof]
    usage_proofs: list[UsageProof]
    correct_lancedb_path: str
    selected_tables: list[str] = Field(default_factory=list)
    missing_files: list[str] = Field(default_factory=list)
    rejected_claims: list[str] = Field(default_factory=list)

class TestResult(BaseModel):
    test_id: str = Field(default_factory=lambda: f"test_{uuid.uuid4().hex[:8]}")
    test_name: str
    passed: bool
    command: Optional[str] = None
    stdout: str = ""
    stderr: str = ""
    returncode: Optional[int] = None
    proof_files: list[str] = Field(default_factory=list)
    error: Optional[str] = None

print("Schema protection loaded.")

Schema protection loaded.


In [4]:
# Cell 4 — Ray boot, list actors, spin down actors safely

import ray

RAY_NAMESPACE = "legion"

def start_ray(local_fallback: bool = True):
    if ray.is_initialized():
        print("Ray already initialized.")
        return

    try:
        ray.init(address="auto", namespace=RAY_NAMESPACE, ignore_reinit_error=True)
        print("Connected to existing Ray cluster, namespace:", RAY_NAMESPACE)
    except Exception as e:
        print("Could not connect to existing Ray cluster:", e)
        if not local_fallback:
            raise
        ray.init(namespace=RAY_NAMESPACE, ignore_reinit_error=True)
        print("Started local Ray, namespace:", RAY_NAMESPACE)

def list_live_actors():
    try:
        from ray.util.state import list_actors
        actors = list_actors(filters=[("ray_namespace", "=", RAY_NAMESPACE)])
        print(f"Live actors in namespace '{RAY_NAMESPACE}': {len(actors)}")
        for a in actors:
            print({
                "name": a.get("name"),
                "state": a.get("state"),
                "class_name": a.get("class_name"),
                "actor_id": a.get("actor_id"),
            })
        return actors
    except Exception as e:
        print("Could not use ray.util.state.list_actors:", e)
        print("Fallback: use `ray list actors` in terminal.")
        return []

def kill_named_actor(name: str):
    try:
        actor = ray.get_actor(name, namespace=RAY_NAMESPACE)
        ray.kill(actor, no_restart=True)
        print("Killed actor:", name)
    except Exception as e:
        print(f"Could not kill actor {name}:", e)

def shutdown_local_ray_only():
    if ray.is_initialized():
        ray.shutdown()
        print("Notebook disconnected from Ray.")

start_ray()
actors_now = list_live_actors()

2026-07-10 22:01:32,450	INFO worker.py:1833 -- Connecting to existing Ray cluster at address: 127.0.0.1:6379...
2026-07-10 22:01:32,481	INFO worker.py:2015 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


Connected to existing Ray cluster, namespace: legion
Live actors in namespace 'legion': 0


In [5]:
# Cell 5 — LanceDB truth probe

import lancedb

def probe_lancedb(path: Path):
    out = {"path": str(path), "exists": path.exists(), "tables": [], "error": None}
    try:
        db = lancedb.connect(str(path))
        out["tables"] = db.table_names()
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

lancedb_probes = [
    probe_lancedb(CORRECT_LANCEDB_PATH),
    probe_lancedb(WRONG_EMPTY_E_PATH),
    probe_lancedb(PROJECT_ROOT / "lancedb_web_intel_rag"),
    probe_lancedb(PROJECT_ROOT / "Legion-Jacked-Pipeline" / "ableton-session-intelligence" / "lancedb_web_intel_rag"),
]

print(json.dumps(lancedb_probes, indent=2))

[
  {
    "path": "C:\\STUDIES_BACKUP\\Legion-Jacked-Pipeline\\ableton-session-intelligence\\lancedb_web_intel_rag",
    "exists": true,
    "tables": [
      "chris_lake_speed_test",
      "chris_lake_web_intel",
      "mined_code_vectors",
      "mined_documentation_vectors"
    ],
    "error": null
  },
  {
    "path": "E:\\WEB CASE STUDY\\Legion-Jacked-Pipeline\\ableton-session-intelligence\\lancedb_web_intel_rag",
    "exists": true,
    "tables": [],
    "error": null
  },
  {
    "path": "C:\\WEB CASE STUDY\\lancedb_web_intel_rag",
    "exists": true,
    "tables": [],
    "error": null
  },
  {
    "path": "C:\\WEB CASE STUDY\\Legion-Jacked-Pipeline\\ableton-session-intelligence\\lancedb_web_intel_rag",
    "exists": true,
    "tables": [],
    "error": null
  }
]


C:\Users\adams\AppData\Local\Temp\ipykernel_14508\1904075430.py:9: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  out["tables"] = db.table_names()


In [7]:
# Cell 6 — Run Adam's local RAG query tool and preserve raw proof

import subprocess
import json
from datetime import datetime
from pathlib import Path

RAG_TOOL = PROJECT_ROOT / "ADAMSCARMCCOY_QUERY_RAG.PY"
PYTHON_EXE = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"

RAG_EVIDENCE_DIR = PROJECT_ROOT / "code_truth_exports" / "rag_evidence"
RAG_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def run_adam_rag(query: str, timeout: int = 180):
    cmd = [str(PYTHON_EXE), str(RAG_TOOL), query]

    p = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_query = "".join(c if c.isalnum() else "_" for c in query.lower())[:80]
    out_path = RAG_EVIDENCE_DIR / f"rag_{stamp}_{safe_query}.txt"

    evidence = {
        "query": query,
        "cmd": " ".join(cmd),
        "returncode": p.returncode,
        "stdout_chars": len(p.stdout or ""),
        "stderr_chars": len(p.stderr or ""),
        "stdout_path": str(out_path),
    }

    out_path.write_text(
        f"QUERY: {query}\n"
        f"COMMAND: {' '.join(cmd)}\n"
        f"RETURNCODE: {p.returncode}\n\n"
        f"STDERR:\n{p.stderr}\n\n"
        f"STDOUT:\n{p.stdout}\n",
        encoding="utf-8",
        errors="replace",
    )

    print("=" * 100)
    print(f"QUERY: {query}")
    print(f"RETURNCODE: {p.returncode}")
    print(f"EVIDENCE SAVED: {out_path}")
    print("=" * 100)

    if p.stderr.strip():
        print("\nSTDERR:")
        print(p.stderr)

    print("\nSTDOUT PREVIEW:")
    print((p.stdout or "")[:6000])

    return evidence, p.stdout or "", p.stderr or ""

evidence, rag_stdout, rag_stderr = run_adam_rag("ray actors ray.get_actor namespace legion")
evidence

: 

In [ ]:
# Kill stale PathVerifierActor if it exists

import ray

try:
    stale = ray.get_actor("PathVerifierActor", namespace="legion")
    ray.kill(stale)
    print("Killed stale PathVerifierActor")
except ValueError:
    print("No stale PathVerifierActor found")

Killed stale PathVerifierActor


In [ ]:
# Cell 7 — Parse RAG evidence into Pydantic code hits

import re
import json
from pathlib import Path
from datetime import datetime
from typing import Optional, List
from pydantic import BaseModel, Field, ValidationError, ConfigDict


class RagCodeHit(BaseModel):
    model_config = ConfigDict(extra="forbid")

    query: str
    result_index: int
    source_file: str
    symbol_name: str
    raw_block: str
    evidence_file: str
    parsed_at: datetime = Field(default_factory=datetime.utcnow)


def parse_rag_code_hits(evidence_path: Path) -> List[RagCodeHit]:
    text = evidence_path.read_text(encoding="utf-8", errors="replace")

    query_match = re.search(r"QUERY:\s*(.+)", text)
    query = query_match.group(1).strip() if query_match else "UNKNOWN_QUERY"

    # Pull only Semantic Code Results area.
    section_match = re.search(
        r"--- Semantic Code Results ---\s*(.*?)(?:--- Semantic Documentation Results ---|Connecting to Ray Swarm Registry|$)",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if not section_match:
        return []

    section = section_match.group(1)

    blocks = re.split(r"\n(?=\[Result #\d+\])", section.strip())
    hits = []

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        idx_match = re.search(r"\[Result #(\d+)\]", block)
        file_match = re.search(r"File:\s*(.+)", block)
        symbol_match = re.search(r"Symbol:\s*(.+)", block)

        if not idx_match or not file_match:
            continue

        try:
            hit = RagCodeHit(
                query=query,
                result_index=int(idx_match.group(1)),
                source_file=file_match.group(1).strip(),
                symbol_name=symbol_match.group(1).strip() if symbol_match else "UNKNOWN_SYMBOL",
                raw_block=block,
                evidence_file=str(evidence_path),
            )
            hits.append(hit)
        except ValidationError as e:
            print(f"Validation failed for block:\n{block[:500]}\nERROR: {e}")

    return hits


latest_evidence = sorted(RAG_EVIDENCE_DIR.glob("rag_*ray_actors_ray_get_actor_namespace_legion.txt"))[-1]
rag_code_hits = parse_rag_code_hits(latest_evidence)

print(f"parsed hits: {len(rag_code_hits)}")
print(json.dumps([h.model_dump(mode="json") for h in rag_code_hits], indent=2)[:5000])

parsed hits: 3
[
  {
    "query": "ray actors ray.get_actor namespace legion",
    "result_index": 1,
    "source_file": "C:/WEB CASE STUDY/mix_audit_agent.py",
    "symbol_name": "__init__",
    "raw_block": "[Result #1] File: C:/WEB CASE STUDY/mix_audit_agent.py\n  Symbol:  __init__\n  Content:\ndef __init__(self):\n        self.registry = ray.get_actor(\"SwarmKnowledgeRegistry\", namespace=LEGION_NAMESPACE)\n        print(\"RegistryClient connected to SwarmKnowledgeRegistry.\")\n...\n--------------------------------------------------",
    "evidence_file": "C:\\WEB CASE STUDY\\code_truth_exports\\rag_evidence\\rag_20260703_023632_ray_actors_ray_get_actor_namespace_legion.txt",
    "parsed_at": "2026-07-03T06:40:36.722361"
  },
  {
    "query": "ray actors ray.get_actor namespace legion",
    "result_index": 2,
    "source_file": "C:/WEB CASE STUDY/legion_sonic_engine_orchestrator.py",
    "symbol_name": "boot_swarm",
    "raw_block": "[Result #2] File: C:/WEB CASE STUDY/legion_sonic

In [ ]:
# Cell 8 — PathVerifierActor: verify RAG source files exist and contain claimed symbols

import ray
from pathlib import Path
from datetime import datetime
from typing import Optional
from pydantic import BaseModel, Field, ConfigDict


class VerifiedCodeHit(BaseModel):
    model_config = ConfigDict(extra="forbid")

    query: str
    result_index: int
    source_file: str
    normalized_path: str
    exists: bool
    readable: bool
    size_bytes: int | None = None
    contains_symbol: bool
    symbol_name: str
    evidence_file: str
    verification_error: str | None = None
    verified_at: datetime = Field(default_factory=datetime.utcnow)


def normalize_windows_path(path_str: str) -> Path:
    # RAG emits C:/WEB CASE STUDY/file.py style paths.
    # Windows accepts both, but normalize anyway because we are not animals.
    return Path(path_str.replace("/", "\\"))


@ray.remote
class PathVerifierActor:
    def verify_hit(self, hit_dict: dict) -> dict:
        try:
            hit = RagCodeHit(**hit_dict)
            path = normalize_windows_path(hit.source_file)

            exists = path.exists()
            readable = False
            size_bytes = None
            contains_symbol = False
            error = None

            if exists and path.is_file():
                size_bytes = path.stat().st_size
                try:
                    text = path.read_text(encoding="utf-8", errors="replace")
                    readable = True

                    if hit.symbol_name and hit.symbol_name != "UNKNOWN_SYMBOL":
                        contains_symbol = hit.symbol_name in text
                    else:
                        contains_symbol = False

                except Exception as e:
                    error = f"{type(e).__name__}: {e}"

            verified = VerifiedCodeHit(
                query=hit.query,
                result_index=hit.result_index,
                source_file=hit.source_file,
                normalized_path=str(path),
                exists=exists,
                readable=readable,
                size_bytes=size_bytes,
                contains_symbol=contains_symbol,
                symbol_name=hit.symbol_name,
                evidence_file=hit.evidence_file,
                verification_error=error,
            )

            return verified.model_dump(mode="json")

        except Exception as e:
            return {
                "query": hit_dict.get("query", "UNKNOWN"),
                "result_index": hit_dict.get("result_index", -1),
                "source_file": hit_dict.get("source_file", "UNKNOWN"),
                "normalized_path": hit_dict.get("source_file", "UNKNOWN"),
                "exists": False,
                "readable": False,
                "size_bytes": None,
                "contains_symbol": False,
                "symbol_name": hit_dict.get("symbol_name", "UNKNOWN"),
                "evidence_file": hit_dict.get("evidence_file", ""),
                "verification_error": f"{type(e).__name__}: {e}",
                "verified_at": datetime.utcnow().isoformat(),
            }


# Reuse detached actor if already alive
path_verifier = PathVerifierActor.options(
    name="PathVerifierActor",
    namespace="legion",
    lifetime="detached",
).remote()

hit_payloads = [h.model_dump(mode="json") for h in rag_code_hits]
verified_refs = [path_verifier.verify_hit.remote(h) for h in hit_payloads]
verified_hits_raw = ray.get(verified_refs)

verified_hits = [VerifiedCodeHit(**v) for v in verified_hits_raw]

print(f"verified hits: {len(verified_hits)}")
print(json.dumps([v.model_dump(mode="json") for v in verified_hits], indent=2)[:8000])

verified hits: 3
[
  {
    "query": "ray actors ray.get_actor namespace legion",
    "result_index": 1,
    "source_file": "C:/WEB CASE STUDY/mix_audit_agent.py",
    "normalized_path": "C:\\WEB CASE STUDY\\mix_audit_agent.py",
    "exists": true,
    "readable": true,
    "size_bytes": 13835,
    "contains_symbol": true,
    "symbol_name": "__init__",
    "evidence_file": "C:\\WEB CASE STUDY\\code_truth_exports\\rag_evidence\\rag_20260703_023632_ray_actors_ray_get_actor_namespace_legion.txt",
    "verification_error": null,
    "verified_at": "2026-07-03T06:41:59.533254"
  },
  {
    "query": "ray actors ray.get_actor namespace legion",
    "result_index": 2,
    "source_file": "C:/WEB CASE STUDY/legion_sonic_engine_orchestrator.py",
    "normalized_path": "C:\\WEB CASE STUDY\\legion_sonic_engine_orchestrator.py",
    "exists": true,
    "readable": true,
    "size_bytes": 5406,
    "contains_symbol": true,
    "symbol_name": "boot_swarm",
    "evidence_file": "C:\\WEB CASE STUDY\\cod

In [ ]:
# Cell 9 — Export verified code truth proof, no tabulate dependency

import pandas as pd
from pathlib import Path
from datetime import datetime
import json

CODE_TRUTH_EXPORTS = PROJECT_ROOT / "code_truth_exports"
CODE_TRUTH_EXPORTS.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

verified_json = CODE_TRUTH_EXPORTS / f"verified_code_hits_{stamp}.json"
verified_csv = CODE_TRUTH_EXPORTS / f"verified_code_hits_{stamp}.csv"
verified_md = CODE_TRUTH_EXPORTS / f"verified_code_hits_{stamp}.md"

rows = [v.model_dump(mode="json") for v in verified_hits]

verified_json.write_text(
    json.dumps(rows, indent=2),
    encoding="utf-8",
    errors="replace",
)

df = pd.DataFrame(rows)
df.to_csv(verified_csv, index=False, encoding="utf-8")

def markdown_escape(value):
    if value is None:
        return ""
    text = str(value)
    text = text.replace("\n", " ").replace("\r", " ")
    text = text.replace("|", "\\|")
    return text

def simple_markdown_table(data_rows, columns):
    if not data_rows:
        return "No verified code hits.\n"

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    lines = [header, sep]

    for row in data_rows:
        line = "| " + " | ".join(markdown_escape(row.get(col, "")) for col in columns) + " |"
        lines.append(line)

    return "\n".join(lines) + "\n"

display_columns = [
    "result_index",
    "source_file",
    "symbol_name",
    "exists",
    "readable",
    "contains_symbol",
    "size_bytes",
    "verification_error",
]

with verified_md.open("w", encoding="utf-8", errors="replace") as f:
    f.write("# Verified Code Hits\n\n")
    f.write(f"Created: {datetime.now().isoformat()}\n\n")
    f.write(f"Total hits: {len(rows)}\n\n")
    f.write(f"Existing files: {sum(1 for r in rows if r.get('exists'))}\n\n")
    f.write(f"Readable files: {sum(1 for r in rows if r.get('readable'))}\n\n")
    f.write(f"Symbol-confirmed files: {sum(1 for r in rows if r.get('contains_symbol'))}\n\n")
    f.write("## Hits\n\n")
    f.write(simple_markdown_table(rows, display_columns))

print("WROTE:")
print(verified_json)
print(verified_csv)
print(verified_md)

WROTE:
C:\WEB CASE STUDY\code_truth_exports\verified_code_hits_20260703_024423.json
C:\WEB CASE STUDY\code_truth_exports\verified_code_hits_20260703_024423.csv
C:\WEB CASE STUDY\code_truth_exports\verified_code_hits_20260703_024423.md


In [ ]:
print(verified_json.exists(), verified_json)
print(verified_csv.exists(), verified_csv)
print(verified_md.exists(), verified_md)

True C:\WEB CASE STUDY\code_truth_exports\verified_code_hits_20260703_024423.json
True C:\WEB CASE STUDY\code_truth_exports\verified_code_hits_20260703_024423.csv
True C:\WEB CASE STUDY\code_truth_exports\verified_code_hits_20260703_024423.md


In [ ]:
# Cell 10A — Build a Source Truth Pack from current verified hits

from pathlib import Path
from datetime import datetime
from typing import List, Optional
from pydantic import BaseModel, Field, ConfigDict
import json
import ray


RAY_NAMESPACE = "legion"
TRUTH_OUT = PROJECT_ROOT / "code_truth_exports" / "truth_packs"
TRUTH_OUT.mkdir(parents=True, exist_ok=True)


class TruthPackLite(BaseModel):
    model_config = ConfigDict(extra="forbid")

    pack_id: str = Field(default_factory=lambda: f"truth_pack_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    created_at: datetime = Field(default_factory=datetime.utcnow)

    queries: list[str]
    correct_lancedb_path: str
    selected_tables: list[str]

    rag_code_hits: list[RagCodeHit]
    verified_code_hits: list[VerifiedCodeHit]

    missing_files: list[str] = Field(default_factory=list)
    unreadable_files: list[str] = Field(default_factory=list)
    unconfirmed_symbols: list[str] = Field(default_factory=list)
    rejected_claims: list[str] = Field(default_factory=list)

    rules_for_codegen: list[str] = Field(default_factory=lambda: [
        "Do not use mock data.",
        "Do not use example.com.",
        "Do not use placeholder rows.",
        "No loose dicts between actors. Use Pydantic models.",
        "Do not trust RAG hits unless PathVerifierActor confirmed file existence.",
        "Generated code is not trusted until TestRunnerActor writes proof.",
    ])


@ray.remote(num_cpus=0.25)
class SourcePackActorLite:
    def build_pack(
        self,
        queries: list[str],
        rag_code_hits_raw: list[dict],
        verified_hits_raw: list[dict],
        lancedb_probes_raw: list[dict],
        correct_lancedb_path: str,
        out_dir: str,
    ) -> dict:

        selected_tables = []
        for probe in lancedb_probes_raw:
            if probe.get("path") == correct_lancedb_path:
                selected_tables = probe.get("tables", [])

        rag_hits = [RagCodeHit.model_validate(h) for h in rag_code_hits_raw]
        verified = [VerifiedCodeHit.model_validate(v) for v in verified_hits_raw]

        missing_files = [v.source_file for v in verified if not v.exists]
        unreadable_files = [v.source_file for v in verified if v.exists and not v.readable]
        unconfirmed_symbols = [
            f"{v.source_file} :: {v.symbol_name}"
            for v in verified
            if v.readable and not v.contains_symbol
        ]

        rejected_claims = []
        for v in verified:
            if not v.exists:
                rejected_claims.append(f"RAG claimed missing file: {v.source_file}")
            elif not v.readable:
                rejected_claims.append(f"RAG file unreadable: {v.source_file}")
            elif not v.contains_symbol:
                rejected_claims.append(f"RAG claimed symbol not confirmed: {v.source_file} :: {v.symbol_name}")

        pack = TruthPackLite(
            queries=queries,
            correct_lancedb_path=correct_lancedb_path,
            selected_tables=selected_tables,
            rag_code_hits=rag_hits,
            verified_code_hits=verified,
            missing_files=missing_files,
            unreadable_files=unreadable_files,
            unconfirmed_symbols=unconfirmed_symbols,
            rejected_claims=rejected_claims,
        )

        out_path = Path(out_dir) / f"{pack.pack_id}.md"
        json_path = Path(out_dir) / f"{pack.pack_id}.json"

        with out_path.open("w", encoding="utf-8", errors="replace") as f:
            f.write(f"# Verified Code Truth Pack: {pack.pack_id}\n\n")
            f.write(f"Created: {pack.created_at.isoformat()}\n\n")

            f.write("## Correct LanceDB Path\n\n")
            f.write(f"`{pack.correct_lancedb_path}`\n\n")

            f.write("## Selected Tables\n\n")
            for t in pack.selected_tables:
                f.write(f"- `{t}`\n")

            f.write("\n## Rules for Codegen\n\n")
            for rule in pack.rules_for_codegen:
                f.write(f"- {rule}\n")

            f.write("\n## Verified Files\n\n")
            for v in pack.verified_code_hits:
                status = "OK" if v.exists and v.readable and v.contains_symbol else "REJECT"
                f.write(f"### {status}: `{v.source_file}`\n\n")
                f.write(f"- normalized_path: `{v.normalized_path}`\n")
                f.write(f"- symbol: `{v.symbol_name}`\n")
                f.write(f"- exists: `{v.exists}`\n")
                f.write(f"- readable: `{v.readable}`\n")
                f.write(f"- contains_symbol: `{v.contains_symbol}`\n")
                f.write(f"- size_bytes: `{v.size_bytes}`\n")
                f.write(f"- verification_error: `{v.verification_error}`\n")
                f.write(f"- evidence_file: `{v.evidence_file}`\n\n")

            f.write("## Rejected Claims\n\n")
            if pack.rejected_claims:
                for claim in pack.rejected_claims:
                    f.write(f"- {claim}\n")
            else:
                f.write("No rejected claims.\n")

            f.write("\n## Raw JSON\n\n```json\n")
            f.write(pack.model_dump_json(indent=2))
            f.write("\n```\n")

        json_path.write_text(
            pack.model_dump_json(indent=2),
            encoding="utf-8",
            errors="replace",
        )

        return {
            "pack_id": pack.pack_id,
            "md_path": str(out_path),
            "json_path": str(json_path),
            "verified_count": len(verified),
            "rejected_claim_count": len(rejected_claims),
        }


# Kill stale SourcePackActorLite during notebook development
try:
    old = ray.get_actor("SourcePackActorLite", namespace=RAY_NAMESPACE)
    ray.kill(old)
    print("Killed stale SourcePackActorLite")
except ValueError:
    pass


pack_actor = SourcePackActorLite.options(
    name="SourcePackActorLite",
    namespace=RAY_NAMESPACE,
    lifetime="detached",
).remote()


queries_for_pack = sorted(set(h.query for h in rag_code_hits))

pack_info = ray.get(
    pack_actor.build_pack.remote(
        queries_for_pack,
        [h.model_dump(mode="json") for h in rag_code_hits],
        [v.model_dump(mode="json") for v in verified_hits],
        lancedb_probes,
        str(CORRECT_LANCEDB_PATH),
        str(TRUTH_OUT),
    )
)

pack_info

{'pack_id': 'truth_pack_20260703_024645',
 'md_path': 'C:\\WEB CASE STUDY\\code_truth_exports\\truth_packs\\truth_pack_20260703_024645.md',
 'json_path': 'C:\\WEB CASE STUDY\\code_truth_exports\\truth_packs\\truth_pack_20260703_024645.json',
 'verified_count': 3,
 'rejected_claim_count': 0}

In [ ]:
# Cell 11 — Optional Gemini CodegenActor using verified pack only

# Use only after the truth pack looks right.

from dotenv import load_dotenv

@ray.remote(num_cpus=0.25)
class GeminiCodegenActor:
    def __init__(self):
        load_dotenv()
        from google import genai
        self.client = genai.Client()

    def generate_from_pack(self, pack_md_path: str, task_prompt: str, out_path: str, model: str = "gemini-2.5-flash") -> dict:
        from google.genai import types

        if not Path(pack_md_path).exists():
            return {"ok": False, "error": f"Pack missing: {pack_md_path}"}

        uploaded = self.client.files.upload(
            file=pack_md_path,
            config=types.UploadFileConfig(mime_type="text/plain", display_name=Path(pack_md_path).name),
        )

        strict_prompt = f"""
Use the uploaded verified source pack as the only source of truth.

Do not use mock data.
Do not use example.com.
Do not use placeholder rows.
No loose dicts between stages.
Use Pydantic v2 schemas for every actor boundary.

Task:
{task_prompt}
"""

        chat = self.client.chats.create(model=model)
        out = Path(out_path)
        with out.open("w", encoding="utf-8") as f:
            response = chat.send_message_stream([uploaded, strict_prompt])
            for chunk in response:
                text = chunk.text or ""
                print(text, end="")
                f.write(text)

        return {"ok": True, "out_path": str(out)}

print("GeminiCodegenActor defined. Do not run until the pack is correct.")

GeminiCodegenActor defined. Do not run until the pack is correct.


In [ ]:
# Cell 12 — TestRunnerActor

@ray.remote(num_cpus=0.5)
class TestRunnerActor:
    def run_command(self, cmd: list[str], cwd: str, test_name: str, timeout: int = 180) -> dict:
        try:
            p = subprocess.run(
                cmd,
                cwd=cwd,
                text=True,
                encoding="utf-8",
                errors="replace",
                capture_output=True,
                timeout=timeout,
            )
            return TestResult(
                test_name=test_name,
                passed=(p.returncode == 0),
                command=" ".join(cmd),
                stdout=p.stdout[-8000:],
                stderr=p.stderr[-8000:],
                returncode=p.returncode,
            ).model_dump()
        except Exception as e:
            return TestResult(
                test_name=test_name,
                passed=False,
                command=" ".join(cmd),
                error=f"{type(e).__name__}: {e}",
            ).model_dump()

    def import_file_test(self, module_name: str, cwd: str) -> dict:
        code = f"import sys; sys.path.insert(0, r'{cwd}'); import {module_name}; print('IMPORT_OK:{module_name}')"
        return self.run_command([str(PYTHON_EXE), "-c", code], cwd, f"import:{module_name}")

    def run_work_finder_test(self, cwd: str) -> dict:
        script = Path(cwd) / "run_work_finder_test.py"
        if not script.exists():
            return TestResult(
                test_name="run_work_finder_test.py",
                passed=False,
                error=f"Missing test script: {script}",
            ).model_dump()
        return self.run_command([str(PYTHON_EXE), str(script)], cwd, "run_work_finder_test.py", timeout=300)

test_runner = TestRunnerActor.options(
    name="TestRunnerActor",
    namespace=RAY_NAMESPACE,
    lifetime="detached"
).remote()

# Example after generated files exist:
# results = ray.get([
#     test_runner.import_file_test.remote("work_lead_schemas", str(PROJECT_ROOT)),
#     test_runner.import_file_test.remote("work_lead_tables", str(PROJECT_ROOT)),
#     test_runner.import_file_test.remote("indie_work_finder_actor", str(PROJECT_ROOT)),
# ])
# print(json.dumps(results, indent=2))

In [ ]:
# Cell 13 — ProofReporterActor

@ray.remote(num_cpus=0.25)
class ProofReporterActor:
    def write_test_report(self, test_results: list[dict], out_dir: str) -> dict:
        import pandas as pd

        results = [TestResult.model_validate(r).model_dump() for r in test_results]
        out = Path(out_dir)
        out.mkdir(parents=True, exist_ok=True)

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        csv_path = out / f"test_results_{stamp}.csv"
        json_path = out / f"test_results_{stamp}.json"
        md_path = out / f"test_results_{stamp}.md"

        df = pd.DataFrame(results)
        df.to_csv(csv_path, index=False)
        json_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

        with md_path.open("w", encoding="utf-8") as f:
            f.write(f"# Code Truth Test Report {stamp}\n\n")
            f.write(f"- total tests: {len(results)}\n")
            f.write(f"- passed: {sum(1 for r in results if r['passed'])}\n")
            f.write(f"- failed: {sum(1 for r in results if not r['passed'])}\n\n")
            for r in results:
                status = "PASS" if r["passed"] else "FAIL"
                f.write(f"## {status}: {r['test_name']}\n\n")
                f.write(f"- command: `{r.get('command')}`\n")
                f.write(f"- returncode: `{r.get('returncode')}`\n")
                if r.get("error"):
                    f.write(f"- error: `{r['error']}`\n")
                if r.get("stderr"):
                    f.write("\n### STDERR\n\n```text\n")
                    f.write(r["stderr"][-2000:])
                    f.write("\n```\n")
                if r.get("stdout"):
                    f.write("\n### STDOUT\n\n```text\n")
                    f.write(r["stdout"][-2000:])
                    f.write("\n```\n")
                f.write("\n")

        return {"csv": str(csv_path), "json": str(json_path), "md": str(md_path)}

proof_reporter = ProofReporterActor.options(
    name="ProofReporterActor",
    namespace=RAY_NAMESPACE,
    lifetime="detached"
).remote()

print("ProofReporterActor ready.")

ProofReporterActor ready.


In [ ]:
# Cell 14 — Orchestration template

def build_truth_pack_for_queries(queries: list[str]):
    retrieved = []
    for q in queries:
        retrieved.extend(ray.get(retriever.query.remote(q)))

    path_proofs = ray.get([verifier.verify.remote(h) for h in retrieved])
    usage_proofs = ray.get([tracer.trace.remote(pp) for pp in path_proofs])

    pack_info = ray.get(
        pack_actor.build_pack.remote(
            queries,
            retrieved,
            path_proofs,
            usage_proofs,
            lancedb_probes,
            str(TRUTH_OUT),
        )
    )
    return pack_info

job_finder_queries = [
    "scrape_prospects download_and_clean weaponize",
    "website scraper prospects contact email outreach pydantic",
    "FastMCP lancedb semantic_code_search",
    "ray actors ray.get_actor namespace legion",
]

# Run this when ready:
# pack_info = build_truth_pack_for_queries(job_finder_queries)
# print(pack_info)

In [ ]:
# Cell 15 — Actor memory control

def kill_truth_swarm():
    for name in [
        "CodeRetrieverActor",
        "PathVerifierActor",
        "UsageTracerActor",
        "SourcePackActor",
        "GeminiCodegenActor",
        "TestRunnerActor",
        "ProofReporterActor",
    ]:
        kill_named_actor(name)

# Inspect:
# list_live_actors()

# Kill truth swarm:
# kill_truth_swarm()

# Disconnect notebook from Ray:
# shutdown_local_ray_only()

In [ ]:
# Cell 16 — Terminal commands reference

print(r"""
Terminal commands you will actually use:

cd "C:\WEB CASE STUDY"

# See Ray status
ray status

# List actors
ray list actors

# Stop local Ray cluster if you intentionally want to nuke local state
ray stop

# Run local RAG query manually
.venv\Scripts\python.exe ADAMSCARMCCOY_QUERY_RAG.PY "scrape_prospects download_and_clean weaponize"

# Start existing MCP RAG server if needed
.venv\Scripts\python.exe mcp_rag_server.py --sse

# Run generated work finder test after the truth swarm approves code
.venv\Scripts\python.exe run_work_finder_test.py
""")


Terminal commands you will actually use:

cd "C:\WEB CASE STUDY"

# See Ray status
ray status

# List actors
ray list actors

# Stop local Ray cluster if you intentionally want to nuke local state
ray stop

# Run local RAG query manually
.venv\Scripts\python.exe ADAMSCARMCCOY_QUERY_RAG.PY "scrape_prospects download_and_clean weaponize"

# Start existing MCP RAG server if needed
.venv\Scripts\python.exe mcp_rag_server.py --sse

# Run generated work finder test after the truth swarm approves code
.venv\Scripts\python.exe run_work_finder_test.py



In [ ]:
import ray

# Connect to the existing local Ray cluster
ray.init(namespace="legion", ignore_reinit_error=True)

# Bind to the specific "good" actor that was holding your state
try:
    # Replace with the actual name of your successful worker
    good_worker = ray.get_actor("IndependentWorkFinderActor") 
    
    # Assuming the actor has a method to return its accepted buffer
    extracted_leads = ray.get(good_worker.get_accepted_leads.remote())
    
    print(f"Successfully extracted {len(extracted_leads)} records directly from hot memory.")
except ValueError:
    print("Actor is no longer alive in memory. Proceeding to Parquet extraction.")

2026-07-03 02:53:06,240	INFO worker.py:1847 -- Calling ray.init() again after it has already been called.


Actor is no longer alive in memory. Proceeding to Parquet extraction.


In [ ]:
import pandas as pd
from pathlib import Path

# Point this to wherever your good workers saved the final state
parquet_path = Path(r"C:\WEB CASE STUDY\work_finder_exports\good_run_data.parquet")

if parquet_path.exists():
    # Load the highly compressed binary data back into a dataframe
    df = pd.read_parquet(parquet_path)
    print(f"Loaded {len(df)} records from Parquet.")
    
    # If you need them back as Pydantic objects for the next strict pipeline phase:
    # records = [WorkLeadRecord(**row) for row in df.to_dict(orient="records")]
else:
    print("Parquet file not found. Check the export path.")

Parquet file not found. Check the export path.
